# Core 03 - Agent

Objetivo: declarar contrato, policy y runtime antes de ejecutar `toolkit.agent`. El provider puede cambiar sin cambiar la interfaz del Agent.

## Parametros de la demostracion

| Variable | Default | Proposito |
|---|---|---|
| AGENTIC_SYSTEMS_AGENT_PROVIDER | python-runtime | Cambiar provider sin cambiar el agente. |
| AGENTIC_SYSTEMS_DEMO_SYMBOL | agent | Entrada que consume la Tool. |
| contract | ContractPolicySpec | Hacer observable la politica de ejecucion. |

In [ ]:
import os

import agentic_systems as toolkit

PROVIDER = "python-runtime"
SYMBOL = os.getenv("AGENTIC_SYSTEMS_DEMO_SYMBOL", "agent")
scheduler = toolkit.scheduler(timeout_s=60, max_retries=0, max_tool_calls=1, max_turns=3)
runtime = toolkit.runtime(provider=PROVIDER, scheduler=scheduler)
toolkit.show_json(runtime.describe(), title="Agent runtime")

## 1) Tool y controles de ejecucion

In [ ]:
@toolkit.tool
def inspect_public_api(symbol: str) -> dict:
    return {"symbol": symbol, "is_public": symbol in toolkit.__all__}

spec = toolkit.ContractPolicySpec(
    name="tutorial.agent.inspect",
    contract=toolkit.AgentContract(
        must_call=["inspect_public_api"],
        completion="when_required_tools_satisfied",
    ),
    policy=toolkit.RunPolicy(max_tool_calls=1, max_turns=3, temperature=0.0),
)
toolkit.show_json(spec.describe(), title="Agent controls")

## 2) Crear y ejecutar el Agent

Python runtime recibe una llamada estructurada. Providers LM reciben una instruccion natural; ambos retornan `RunResult`.

In [ ]:
agent = toolkit.agent(
    name="public_api_agent",
    instructions="Usa inspect_public_api y conserva la evidencia observada.",
    tools=[inspect_public_api],
    runtime=runtime,
    **spec.agent_kwargs(),
)

request = (
    {"tool": "inspect_public_api", "input": {"symbol": SYMBOL}}
    if PROVIDER == "python-runtime"
    else f"Usa inspect_public_api para verificar {SYMBOL}."
)
result = agent.run(request, mode="eval")
toolkit.human_result(result, title="Agent RunResult", show_lineage=True)
toolkit.show_json(toolkit.agent_output(result), title="Canonical agent_output")

## 3) API realmente ejercitada

In [ ]:
api_coverage = [
    "toolkit.scheduler", "toolkit.runtime", "toolkit.tool", "toolkit.ContractPolicySpec",
    "toolkit.AgentContract", "toolkit.RunPolicy", "toolkit.agent", "agent.run",
    "toolkit.human_result", "toolkit.agent_output",
    "toolkit.show_json",
]
toolkit.show_json(api_coverage, title="Agent API coverage")

## Resultado esperado

Un RunResult validado cuya metadata muestra el provider seleccionado y cuya evidencia contiene `inspect_public_api`.